# 🧠 Ciencia de Datos II — EDA Progresivo
### Dataset: Stroke Prediction

Vamos a explorar el dataset de forma ordenada, respondiendo **una pregunta concreta por bloque**.
La regla de oro: **no modificar nada antes de entender**.

---
## BLOQUE 1 — Carga y dimensiones del dataset

**¿Qué hacemos acá?**
Cargamos el archivo CSV y vemos qué tamaño tiene la base.

**¿Por qué importa?**
Antes de cualquier análisis hay que saber con cuántos registros y cuántas variables trabajamos.
Un dataset de 100 filas se analiza distinto a uno de 25.000.

In [ ]:
import pandas as pd

# ============================================================
# BLOQUE 1 — CARGA Y DIMENSIONES DEL DATASET
# ============================================================

stroke = pd.read_csv('https://raw.githubusercontent.com/MarisaBerger/ciencia-datos-II/main/data/dataset_04_stroke.csv')

filas, columnas = stroke.shape
print(f"Filas    : {filas}")
print(f"Columnas : {columnas}")

**¿Qué aprendemos?**
El dataset tiene **25.000 pacientes** y **18 variables**.
Esto nos da una primera idea del tamaño de la muestra antes de hacer cualquier análisis.

---
## BLOQUE 2 — Variables del dataset

**¿Qué hacemos acá?**
Listamos los nombres de todas las columnas.

**¿Por qué importa?**
Los nombres ya nos cuentan cosas: podemos detectar variables que fueron codificadas,
variables dummy (one-hot encoding) y la variable objetivo (`target`).

In [ ]:
# ============================================================
# BLOQUE 2 — VARIABLES DEL DATASET
# ============================================================

print("Columnas del dataset:")
for i, col in enumerate(stroke.columns, 1):
    print(f"  {i:2d}. {col}")

**¿Qué aprendemos?**
Se observan dos grupos de variables claramente codificadas:

- `work_type_Govt_job`, `work_type_Private`, `work_type_Self-employed`... → probablemente provienen de una variable original `work_type` convertida a **dummies**
- `smoking_status_Unknown`, `smoking_status_formerly smoked`... → ídem con `smoking_status`

También aparecen variables como `gender_encoded`, `ever_married_encoded`, `Residence_type_encoded` que ya fueron transformadas numéricamente.

Esto nos indica que el dataset fue **preprocesado antes de entregarnos**. Hay que estudiarlo con cuidado.

---
## BLOQUE 3 — Primeras observaciones

**¿Qué hacemos acá?**
Miramos las primeras filas de la tabla.

**¿Por qué importa?**
No es solo "ver la tabla". Sirve para detectar rápidamente:
- cómo están expresados los datos
- presencia de `NaN`
- cantidad de decimales
- variables con valores que parecen raros a primera vista

In [ ]:
# ============================================================
# BLOQUE 3 — PRIMERAS OBSERVACIONES
# ============================================================

stroke.head(10)

**¿Qué aprendemos?**
Ya en las primeras filas se nota algo importante: variables como `hypertension` o `heart_disease`,
que por su nombre uno esperaría que fueran **binarias (0 o 1)**, muestran valores **decimales, negativos y mayores a 1**.

Esto no significa automáticamente que estén mal. Lo anotamos como:

> ⚠️ **Hallazgo a investigar:** variables aparentemente binarias tienen valores continuos.

---
## BLOQUE 4 — Estructura y tipo de dato

**¿Qué hacemos acá?**
Vemos el tipo de dato que Python/pandas asignó a cada columna.

**¿Por qué importa?**
Hay una distinción fundamental que hay que defender:

> **Tipo de dato en pandas ≠ significado estadístico de la variable**

Una columna puede estar guardada como `float64` (numérica) y ser conceptualmente **categórica**.

In [ ]:
# ============================================================
# BLOQUE 4 — ESTRUCTURA Y TIPO DE DATO
# ============================================================

# Información general: tipo de dato, valores no nulos
stroke.info()

In [ ]:
# Ver solo los tipos de dato en formato tabla
pd.DataFrame({
    'Tipo de dato': stroke.dtypes,
    'Valores no nulos': stroke.notna().sum(),
    'Valores nulos': stroke.isna().sum()
})

**¿Qué aprendemos?**
Prácticamente todas las variables son `float64`. Pero eso no significa que todas sean cuantitativas continuas.
Por ejemplo, `target` es numérica pero conceptualmente es **binaria categórica** (0 = sin ACV, 1 = con ACV).

---
## BLOQUE 5 — Clasificación conceptual de las variables

Antes de calcular medias y gráficos, clasificamos cada variable según **qué representa**, no según cómo está guardada.

| Variable | Significado probable | Tipo conceptual |
|---|---|---|
| `age` | Edad | Cuantitativa continua |
| `hypertension` | Hipertensión | Originalmente binaria (transformada) |
| `heart_disease` | Enfermedad cardíaca | Originalmente binaria (transformada) |
| `avg_glucose_level` | Glucemia promedio | Cuantitativa continua |
| `bmi` | Índice de masa corporal | Cuantitativa continua |
| `gender_encoded` | Género | Categórica codificada |
| `ever_married_encoded` | Estado civil | Categórica/binaria codificada |
| `Residence_type_encoded` | Tipo de residencia | Categórica codificada |
| `work_type_*` | Tipo de trabajo | Dummy (one-hot encoding) |
| `smoking_status_*` | Estado tabáquico | Dummy (one-hot encoding) |
| `target` | Presencia de ACV | Binaria (variable objetivo) |

> ⚠️ **Importante:** calcular la media de `gender_encoded` no tiene el mismo sentido fisiológico que calcular la media de `bmi`. Hay que ser cuidadosas con qué interpretamos.

---
## BLOQUE 6 — Valores faltantes

**¿Qué hacemos acá?**
Contamos cuántos `NaN` tiene cada columna, y calculamos el porcentaje.

**¿Por qué importa?**
Un faltante no se analiza solo contando cuántos hay.
Hay que ver si **se concentra en determinados grupos** o si está distribuido al azar.
También hay que buscar "**faltantes disfrazados**": valores como -999, 0 o 999 que en realidad representan "sin dato".

In [ ]:
# ============================================================
# BLOQUE 6 — DATOS FALTANTES
# ============================================================

faltantes = pd.DataFrame({
    'Variable': stroke.columns,
    'Cantidad_NA': stroke.isna().sum().values,
    'Porcentaje_NA': (stroke.isna().mean() * 100).round(2).values
}).sort_values('Cantidad_NA', ascending=False).reset_index(drop=True)

faltantes

**¿Qué aprendemos?**
Varias columnas tienen **exactamente 5.500 faltantes**, lo que representa el **22%** de las observaciones.

Esto no parece casualidad. La pregunta que surge inmediatamente es:

> **¿Son los mismos 5.500 pacientes los que tienen `NaN` en todas esas columnas?**

Si lo fueran, hay un **patrón sistemático de faltantes** que hay que investigar.
Eso lo vamos a ver en el Bloque 9 (análisis detallado de NA).

Por ahora lo anotamos como hallazgo.

---
## BLOQUE 7 — Registros duplicados

**¿Qué hacemos acá?**
Buscamos filas completamente iguales en todas sus columnas.

**¿Por qué importa?**
Un duplicado exacto puede indicar un error al armar el dataset (por ejemplo, un registro cargado dos veces).
Si aparecen muchos, habría que evaluar si eliminarlos o investigar por qué están.

In [ ]:
# ============================================================
# BLOQUE 7 — REGISTROS DUPLICADOS
# ============================================================

duplicados = stroke.duplicated().sum()
print(f"Filas completamente duplicadas: {duplicados}")

**¿Qué aprendemos?**
No hay filas completamente duplicadas.

> ⚠️ **Atención con la expresión "completamente"**: significa que no existe otra fila idéntica en las 18 columnas.
No significa que no pueda haber dos personas con la misma edad, el mismo BMI o la misma glucosa.
Eso sería coincidencia de datos, no un duplicado.

---
## BLOQUE 8 — Distribución de la variable objetivo (`target`)

**¿Qué hacemos acá?**
Contamos cuántos casos hay de cada valor de `target`.

**¿Por qué importa?**
`target` es la variable que queremos predecir (0 = sin ACV, 1 = con ACV).
Si está muy desbalanceada, una simple *accuracy* puede ser engañosa:
un modelo que predice "0" siempre podría tener 87% de accuracy sin aprender nada útil.

In [ ]:
# ============================================================
# BLOQUE 8 — DISTRIBUCIÓN DE LA VARIABLE OBJETIVO
# ============================================================

conteo = stroke['target'].value_counts().rename({0: 'Sin ACV (0)', 1: 'Con ACV (1)'})
porcentaje = (stroke['target'].value_counts(normalize=True) * 100).round(2).rename({0: 'Sin ACV (0)', 1: 'Con ACV (1)'})

resumen_target = pd.DataFrame({
    'Casos': conteo,
    'Porcentaje (%)': porcentaje
})

print(resumen_target)
print(f"\nRelación aproximada: {round(conteo[0] / conteo[1], 1)} casos sin ACV por cada caso con ACV")

**¿Qué aprendemos?**

| Clase | Casos | % |
|---|---|---|
| Sin ACV (0) | 21.875 | 87,5% |
| Con ACV (1) | 3.125 | 12,5% |

El dataset está **desbalanceado**: hay aproximadamente 7 pacientes sin ACV por cada paciente con ACV.

Esto es importante para modelos de Machine Learning: una accuracy alta no garantiza un buen modelo.
Métricas como precisión, recall o F1-score van a ser más relevantes.

---

## ✅ Cierre del primer bloque de análisis

Con estos 8 bloques ya podemos afirmar:

> *"El dataset tiene 25.000 registros y 18 variables. Incluye variables demográficas, antecedentes clínicos, medidas fisiológicas y variables categóricas previamente codificadas. No presenta filas completamente duplicadas. Varias columnas presentan un 22% de datos faltantes con un patrón a investigar. La variable objetivo está desbalanceada: 87,5% sin ACV vs 12,5% con ACV."*

**Lo que sigue:**
- Bloque 9: Resumen estadístico de variables clínicas (`age`, `bmi`, `avg_glucose_level`)
- Bloque 10: Detección de valores extremos y evaluación fisiológica
- Bloque 11: Análisis del patrón de faltantes (¿son los mismos 5.500 pacientes?)